# Historical Backtesting

This notebook keeps the same workflow: load prices, calculate features, form monthly
portfolios, compare models, compare buffers, and estimate transaction costs.

Use 2021–2024 for calibration and 2025 onward for exploratory validation. The latter
has already been inspected when choosing strategies, so it is not an untouched test.
The current asset list and revised provider prices also limit historical realism.

In [ ]:
# Imports

from pathlib import Path
import sys


import numpy as np
import pandas as pd

from sqlalchemy import text


# Project root

PROJECT_ROOT = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "src" / "database.py").is_file()),
    None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the project folder.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )


# Project modules

from src.database import get_engine
from src.features import calculate_features
from src.screener import (
    build_universe,
    select_candidates,
    select_candidates_with_buffer
)
from src.portfolio import calculate_portfolio_weights
from src.backtest import simulate_period

In [ ]:
# Database connection and prices

engine = get_engine()

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))

print("Database connection successful.")


prices = pd.read_sql(
    """
    SELECT
        p.date,
        p.asset_id,
        a.actinver_symbol,
        a.asset_type,
        p.close,
        p.adjusted_close,
        p.volume

    FROM public.prices p

    JOIN public.assets a
        ON p.asset_id = a.asset_id

    ORDER BY
        p.asset_id,
        p.date;
    """,
    engine
)

prices["date"] = pd.to_datetime(
    prices["date"]
)

print(f"Rows: {len(prices):,}")
print(
    f"Assets: "
    f"{prices['asset_id'].nunique()}"
)
print(
    f"Period: "
    f"{prices['date'].min().date()} "
    f"to {prices['date'].max().date()}"
)

# Pending issues are dated by the affected market date, not copied to all past dates.
rejections = pd.read_sql(
    "SELECT asset_id, date FROM public.price_rejections", engine
)
rejections["date"] = pd.to_datetime(rejections["date"])
if rejections["date"].isna().any():
    raise ValueError("Resolve undated price rejections before running the backtest.")
engine.dispose()

## 1. Historical features

The same feature-engineering methodology used by the current screener is used
throughout the backtest.

The model includes:

- 5-observation momentum
- 20-observation momentum
- 60-observation momentum
- 20-observation annualized volatility
- 252-observation current drawdown
- recent trading-activity indicators
- recent extreme-return filters

Using the same functions in both the current screener and historical backtest
ensures methodological consistency.

In [ ]:
# Feature calculation

prices = calculate_features(
    prices
)

In [ ]:
# Backtest configuration

BACKTEST_START = pd.Timestamp(
    "2021-01-01"
)

CALIBRATION_END = pd.Timestamp(
    "2024-12-31"
)

VALIDATION_START = pd.Timestamp(
    "2025-01-01"
)

ENTRY_CUTOFF = 0.90


SCORE_MODELS = {
    "Baseline": {
        "momentum": 1 / 3,
        "volatility": 1 / 3,
        "drawdown": 1 / 3
    },

    "Defensive": {
        "momentum": 0.30,
        "volatility": 0.45,
        "drawdown": 0.25
    }
}


WEIGHTING_METHODS = [
    "equal",
    "category_inverse_volatility",
    "category_score_volatility"
]

In [ ]:
# Rebalance dates: only completed calendar months generate a signal.
market_dates = pd.DatetimeIndex(prices["date"].dropna().unique()).sort_values()
if market_dates.empty:
    raise ValueError("Run notebook 01 first: no prices are available.")

last_complete_month = min(
    market_dates.max().to_period("M"), pd.Timestamp.today().to_period("M")
) - 1
months = pd.period_range(BACKTEST_START.to_period("M"), last_complete_month, freq="M")
rebalance_dates = months.to_timestamp(how="end").normalize()

## 2. Backtest rules

1. Build a signal at calendar month-end. Start placing orders on the next market day.
2. Give each asset its target share of the portfolio's value at this rebalance.
3. If an asset has no valid quote, wait. Keep cash for a pending buy and keep the
   existing position for a pending sale. Never trade at a filled-in price.
4. Each day, execute available sells first, then buys using only available cash.
   If cash is insufficient, share it proportionally across executable buys.
5. A new monthly rebalance replaces unfinished orders. No candidates means sell
   existing holdings when their quotes become available, then remain in cash.
6. Value a held asset with its last observed quote when today's quote is missing.
   This is a valuation estimate, not an executable price. `oldest_quote_days`
   shows the age of the oldest holding quote at the end of each period.

The next rebalance close ends the old period and starts the new one. Cash earns 0%.
The daily execution helper is in `src/backtest.py`: read its sell loop, then its buy
loop. It uses fractional adjusted-price units to model total returns, not actual
share counts. Fees reduce cash when each trade occurs.

Pending rejections exclude an asset from their affected market date onward until
repaired. The database is a current snapshot, not a point-in-time archive.

In [ ]:
# Period return

def calculate_period_return(start_value, end_value):
    """Compare portfolio values, including cash and actual execution costs."""
    if not np.isfinite(start_value) or start_value <= 0 or not np.isfinite(end_value):
        raise ValueError("Portfolio values must be finite, with positive starting capital.")
    return end_value / start_value - 1

In [ ]:
# Main backtest function

def run_backtest(
    prices, rebalance_dates, score_weights, weighting_method,
    entry_cutoff=0.90, use_buffer=False, exit_cutoff=0.80,
    rejections=None, cost_rate=0.0
):
    results = []
    trade_history = []
    holdings = {}             # asset_id -> units actually held
    cash = 1.0               # Start with one unit of capital.
    last_quotes = {}          # Used only to value existing holdings.
    quote_table = prices.pivot(index="date", columns="asset_id", values="adjusted_close").sort_index()

    schedule = []
    for signal_date in rebalance_dates:
        next_dates = quote_table.index[quote_table.index > signal_date]
        if len(next_dates):
            schedule.append((pd.Timestamp(signal_date), next_dates[0]))
    if len(schedule) < 2:
        raise ValueError("At least two monthly execution dates are needed.")

    for i in range(len(schedule) - 1):
        signal_date, entry_date = schedule[i]
        end_date = schedule[i + 1][1]
        if entry_date >= end_date:
            raise ValueError("Missing market history between monthly signals.")

        universe = build_universe(
            prices=prices, evaluation_date=signal_date,
            momentum_weight=score_weights["momentum"],
            volatility_weight=score_weights["volatility"],
            drawdown_weight=score_weights["drawdown"], rejections=rejections
        )
        if use_buffer:
            candidates = select_candidates_with_buffer(
                universe, previous_assets=set(holdings),
                entry_cutoff=entry_cutoff, exit_cutoff=exit_cutoff
            )
        else:
            candidates = select_candidates(universe, cutoff=entry_cutoff)
        candidates = calculate_portfolio_weights(candidates, method=weighting_method)
        candidates = candidates[candidates["weight"] > 0].copy()
        target_weights = candidates.set_index("asset_id")["weight"].to_dict()

        # Only real quotes execute orders. The helper carries cash and unfilled sales.
        active_assets = sorted(set(holdings) | set(target_weights))
        period_quotes = quote_table.loc[entry_date:end_date, active_assets]
        period = simulate_period(
            period_quotes, target_weights, holdings, cash, last_quotes, cost_rate
        )
        holdings = period["holdings"]
        cash = period["cash"]
        last_quotes = period["last_quotes"]
        results.append(dict(
            signal_date=signal_date, entry_date=entry_date, end_date=end_date,
            n_assets=len(holdings), cash_weight=cash / period["end_value"],
            pending_orders=period["pending_orders"], waiting_days=period["waiting_days"],
            oldest_quote_days=period["oldest_quote_days"],
            **{"return": calculate_period_return(period["start_value"], period["end_value"])}
        ))
        for trade in period["trades"]:
            trade_history.append(dict(
                trade, signal_date=signal_date, entry_date=entry_date,
                traded_fraction=trade["notional"] / period["start_value"]
            ))

    backtest = pd.DataFrame(results)
    trades = pd.DataFrame(trade_history, columns=[
        "signal_date", "entry_date", "date", "asset_id", "side", "units", "price",
        "notional", "fee", "traded_fraction"
    ])
    return backtest, trades

In [ ]:
# Metrics

def calculate_metrics(
    backtest
):

    data = backtest.sort_values("entry_date").copy()
    if not np.isfinite(data["return"]).all() or data["return"].le(-1).any():
        raise ValueError("Metrics require valid returns for every period.")
    if len(data) > 1 and not (
        data["entry_date"].iloc[1:].to_numpy() == data["end_date"].iloc[:-1].to_numpy()
    ).all():
        raise ValueError("Backtest periods must be consecutive: a period is missing.")

    if data.empty:
        return {
            "Final Wealth": np.nan,
            "CAGR": np.nan,
            "Annualized Volatility": np.nan,
            "Sharpe Ratio": np.nan,
            "Maximum Drawdown": np.nan
        }

    wealth = (
        1 + data["return"]
    ).cumprod()

    final_wealth = (
        wealth.iloc[-1]
    )

    years = (data["end_date"].iloc[-1] - data["entry_date"].iloc[0]).days / 365.25
    if years <= 0:
        raise ValueError("The measurement period must have positive duration.")

    cagr = (
        final_wealth
        ** (1 / years)
        - 1
    )

    return_std = (
        data["return"].std()
    )

    annualized_volatility = (
        return_std
        * np.sqrt(12)
    )

    if (
        pd.isna(return_std)
        or np.isclose(return_std, 0)
    ):
        sharpe = np.nan

    else:
        sharpe = (
            data["return"].mean()
            / return_std
            * np.sqrt(12)
        )

    wealth_with_initial = pd.concat(
        [
            pd.Series([1.0]),
            wealth.reset_index(drop=True)
        ],
        ignore_index=True
    )

    running_max = (
        wealth_with_initial.cummax()
    )

    drawdown = (
        wealth_with_initial
        / running_max
        - 1
    )

    maximum_drawdown = (
        drawdown.min()
    )

    return {
        "Final Wealth":
            final_wealth,

        "CAGR":
            cagr,

        "Annualized Volatility":
            annualized_volatility,

        "Sharpe Ratio":
            sharpe,

        "Maximum Drawdown":
            maximum_drawdown
    }

In [ ]:
# Turnover: actual purchases plus sales, divided by starting period capital.
# First purchase = 1.0 without fees. A full replacement is approximately 2.0.

def calculate_turnover(trade_history):
    return trade_history.groupby("entry_date")["traded_fraction"].sum().sort_index()

## 3. Score-model calibration

Two economically motivated factor specifications are evaluated.

### Baseline

$$
Score_i
=
\frac13 Momentum_i
-
\frac13 Volatility_i
+
\frac13 Drawdown_i.
$$

### Defensive

$$
Score_i
=
0.30\,Momentum_i
-
0.45\,Volatility_i
+
0.25\,Drawdown_i.
$$

The 2021--2024 period is used for calibration, while 2025 onward is used
for exploratory validation. This period has already influenced model selection.

The objective is not to maximize historical performance over a large parameter
grid, but to compare a small number of economically interpretable
specifications.

In [ ]:
# Run main strategy combinations

backtests = {}
trade_histories = {}


for model_name, score_weights in (
    SCORE_MODELS.items()
):

    for weighting_method in (
        WEIGHTING_METHODS
    ):

        strategy_name = (
            f"{model_name} - "
            f"{weighting_method}"
        )

        print(f"Running: {strategy_name}", flush=True)
        bt, trades = run_backtest(
            prices=prices,
            rejections=rejections,
            rebalance_dates=rebalance_dates,
            score_weights=score_weights,
            weighting_method=weighting_method,
            entry_cutoff=ENTRY_CUTOFF,
            use_buffer=False
        )


        backtests[
            strategy_name
        ] = bt

        trade_histories[
            strategy_name
        ] = trades

In [ ]:
# Calibration vs Validation

comparison_results = []


for strategy_name, bt in (
    backtests.items()
):

    periods = {
        "Calibration 2021-2024":
            bt[
                bt["end_date"]
                <= CALIBRATION_END
            ],

        "Exploratory validation 2025 onward":
            bt[
                bt["entry_date"]
                >= VALIDATION_START
            ]
    }

    for period_name, period_bt in (
        periods.items()
    ):

        metrics = calculate_metrics(
            period_bt
        )

        metrics["Period"] = (
            period_name
        )

        metrics["Strategy"] = (
            strategy_name
        )

        comparison_results.append(
            metrics
        )


strategy_comparison = (
    pd.DataFrame(
        comparison_results
    )
    .set_index(
        [
            "Period",
            "Strategy"
        ]
    )
)


display(
    strategy_comparison[
        [
            "Final Wealth",
            "CAGR",
            "Annualized Volatility",
            "Sharpe Ratio",
            "Maximum Drawdown"
        ]
    ]
)

In [ ]:
# Turnover comparison: match costs to the same periods used for returns.
turnover_results = []
for strategy_name, trades in trade_histories.items():
    bt = backtests[strategy_name]
    turnover = calculate_turnover(trades).reindex(bt["entry_date"], fill_value=0)
    turnover_results.append({
        "Strategy": strategy_name,
        "Calibration Traded Fraction": turnover.to_numpy()[bt["end_date"] <= CALIBRATION_END].mean()
            if (bt["end_date"] <= CALIBRATION_END).any() else np.nan,
        "Validation Traded Fraction": turnover.to_numpy()[bt["entry_date"] >= VALIDATION_START].mean()
            if (bt["entry_date"] >= VALIDATION_START).any() else np.nan
    })
turnover_comparison = pd.DataFrame(turnover_results).set_index("Strategy")
turnover_comparison

## 4. Selected strategy specification

We retain the previously chosen Defensive V1 specification for comparison.
This is a provisional choice, not evidence from an independent test.

The selected V1 score is:

$$
Score_i
=
0.30\,Momentum_i
-
0.45\,Volatility_i
+
0.25\,Drawdown_i
$$

Assets are selected when:

$$
CategoryPercentile_i \geq 0.90.
$$

The preferred portfolio-weighting method is
`category score/volatility`.

For category $c$, the category budget is proportional to the number of
selected assets in that category:

$$
W_c
=
\frac{N_c}{N}.
$$

Within each category:

$$
\tilde w_{i|c}
=
\frac{\max(Score_i,0)}
{\sigma_i},
$$

and:

$$
w_{i|c}
=
\frac{\tilde w_{i|c}}
{\sum_{j\in c}\tilde w_{j|c}}.
$$

The final asset weight is:

$$
w_i
=
W_c w_{i|c}.
$$

If a category has no usable weighting signal, split its existing category budget equally.

In [ ]:
# Buffer experiment

defensive_weights = (
    SCORE_MODELS["Defensive"]
)


buffer_configs = {
    "90/90 No Buffer": {
        "use_buffer": False,
        "exit_cutoff": 0.90
    },

    "90/85 Buffer": {
        "use_buffer": True,
        "exit_cutoff": 0.85
    },

    "90/80 Buffer": {
        "use_buffer": True,
        "exit_cutoff": 0.80
    }
}


buffer_backtests = {}
buffer_trades = {}


for name, config in (
    buffer_configs.items()
):

    print(f"Running buffer: {name}", flush=True)
    bt, trades = run_backtest(
        prices=prices,
        rejections=rejections,
        rebalance_dates=rebalance_dates,
        score_weights=defensive_weights,
        weighting_method=
            "category_score_volatility",
        entry_cutoff=0.90,
        use_buffer=config[
            "use_buffer"
        ],
        exit_cutoff=config[
            "exit_cutoff"
        ]
    )


    buffer_backtests[name] = bt
    buffer_trades[name] = trades

In [ ]:
# Buffer Validation comparison

buffer_results = []


for name, bt in (
    buffer_backtests.items()
):

    validation_bt = bt[
        bt["entry_date"]
        >= VALIDATION_START
    ]

    metrics = calculate_metrics(
        validation_bt
    )

    turnover = calculate_turnover(
        buffer_trades[name]
    ).reindex(bt["entry_date"], fill_value=0)

    validation_turnover = turnover[
        turnover.index
        >= VALIDATION_START
    ]

    metrics["Strategy"] = name
    metrics["Validation Turnover"] = (
        validation_turnover.mean()
    )

    buffer_results.append(
        metrics
    )


buffer_comparison = (
    pd.DataFrame(
        buffer_results
    )
    .set_index("Strategy")
)


buffer_comparison[
    [
        "CAGR",
        "Annualized Volatility",
        "Sharpe Ratio",
        "Maximum Drawdown",
        "Validation Turnover"
    ]
]

### Buffer interpretation

Compare the return, risk, and traded-capital tables after rerunning this notebook.
Do not assume a buffer improves results. V1 keeps the existing no-buffer rule;
these comparisons are exploratory and do not establish an independently tested winner.

In [ ]:
# Cost scenarios must rerun execution: fees change cash available for later buys.

def run_cost_scenario(cost_rate):
    return run_backtest(
        prices=prices, rebalance_dates=rebalance_dates,
        score_weights=SCORE_MODELS["Defensive"],
        weighting_method="category_score_volatility",
        entry_cutoff=ENTRY_CUTOFF, use_buffer=False,
        rejections=rejections, cost_rate=cost_rate
    )

In [ ]:
# Cost sensitivity for the existing provisional V1 specification.
FINAL_STRATEGY = "Defensive - category_score_volatility"
final_bt = backtests[FINAL_STRATEGY]
final_trades = trade_histories[FINAL_STRATEGY]

cost_scenarios = {"Gross": 0.0, "10 bps": 0.001, "25 bps": 0.0025}
cost_results = []
for scenario, cost_rate in cost_scenarios.items():
    print(f"Running costs: {scenario}", flush=True)
    if cost_rate == 0:
        net_bt = final_bt
    else:
        net_bt, _ = run_cost_scenario(cost_rate)
    metrics = calculate_metrics(net_bt)
    metrics["Scenario"] = scenario
    cost_results.append(metrics)
display(pd.DataFrame(cost_results).set_index("Scenario"))

# Inspect delayed orders and stale valuations instead of hiding them.
display(final_bt.loc[
    (final_bt["waiting_days"] > 0) | (final_bt["oldest_quote_days"] > 0),
    ["entry_date", "end_date", "pending_orders", "waiting_days", "cash_weight", "oldest_quote_days"]
])
display(final_trades.head(20))

## 5. Interpretation and limitations

- V1 retains the Defensive score, category score/volatility weights, and no buffer.
- Missing quotes delay trades. Last-known prices value existing holdings only.
  Review `waiting_days`, `pending_orders`, `cash_weight`, and `oldest_quote_days`.
  A long-unquoted holding may overstate portfolio value; delistings require separate
  investigation. It is not assumed to be sellable or silently removed.
- Monthly target budgets are fixed at rebalance. Desired units are set at each
  asset's first executable quote. Unfinished orders are replaced next month.
- Fees are charged on actual daily trades and affect subsequent cash availability.
  Positions are marked to market at the end, without an assumed final liquidation.
- Adjusted-price units, fractional quantities, unlimited volume, and close-price
  execution are simplifying assumptions. This is not a real brokerage simulation.
- The 2025 onward comparisons are validation already used during development.
  Freeze the specification before evaluating a new, unobserved period.
- Today's asset list creates survivorship bias. Revised provider history is not
  a point-in-time dataset. Cash earns 0%, and Sharpe assumes a 0% risk-free rate.
- Risk metrics use monthly values, so intra-month losses are not shown.